# Chapter 01 — One Tool Call

A single tool call is the atomic unit of every agent. This notebook walks through the four-step cycle using the MiniMax M2.7 API.

**Prerequisites:**
```bash
uv add anthropic python-dotenv
```

Store your API key in `.env`: `MINIMAX_API_KEY=your_key_here`

In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv
import json

load_dotenv()
client = Anthropic()

---

## 1. The Four-Step Cycle

```
App → LLM:  user message + tool schemas
LLM → App:  tool_use block (a request, not a call)
App → Tool: execute(name, arguments)
Tool → App: result
App → LLM:  tool_result message
LLM → App:  final answer
```

### Step 1 — Define the tool schema

The schema has three parts the model sees: **name**, **description**, and **input_schema**. The description must tell the model *when not to use* the tool as well as when to use it.

In [2]:
# Tool handlers — real Python functions, registered by name
def get_weather(location: str) -> str:
    """Returns current weather for a single city."""
    return f"24°C, sunny in {location}"

def get_population(city: str) -> str:
    """Returns population for a single city."""
    populations = {
        "Tokyo": "37 million",
        "San Francisco": "880 thousand",
        "New York": "8.3 million",
    }
    return populations.get(city, f"Unknown population for {city}")

# Tool registry — maps tool name → handler function
TOOL_HANDLERS = {
    "get_weather": get_weather,
    "get_population": get_population,
}

tools = [
    {
        "name": "get_weather",
        "description": (
            "Returns current weather conditions for a single city. "
            "Use when the user asks about current weather. "
            "Do NOT use for: historical data, forecasts, or multi-city queries."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and state/country, e.g. 'Tokyo, JP' or 'San Francisco, CA'"
                }
            },
            "required": ["location"]
        }
    },
    {
        "name": "get_population",
        "description": (
            "Returns the population of a single city. "
            "Use when the user asks about a city's population. "
            "Do NOT use for: country-level data, historical populations, or density stats."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name, e.g. 'Tokyo' or 'San Francisco'"
                }
            },
            "required": ["city"]
        }
    }
]

print(f"Registered {len(TOOL_HANDLERS)} tool handlers: {list(TOOL_HANDLERS.keys())}")
print(f"Defined {len(tools)} tool schemas: {[t['name'] for t in tools]}")

Registered 2 tool handlers: ['get_weather', 'get_population']
Defined 2 tool schemas: ['get_weather', 'get_population']


### Step 2 — Send user message and get model's tool_use request

The model returns a list of content blocks. For tool calls, look for `block.type == "tool_use"`. The model emits a request — it cannot execute anything itself.

In [3]:
# Reuse send_messages and process_response from above
def send_messages(messages, tools):
    params = {
        "model": "MiniMax-M2.7",
        "max_tokens": 4096,
        "messages": messages,
    }

    if tools:
        params['tools'] = tools

    response = client.messages.create(**params)
    return response

def process_response(response):
    thinking_blocks = []
    text_blocks = []
    tool_use_blocks = []

    # Iterate through all content blocks
    for block in response.content:
        if block.type == "thinking":
            thinking_blocks.append(block)
            print(f"💭 Thinking>\n{block.thinking}\n")
        elif block.type == "text":
            text_blocks.append(block)
            print(f"💬 Model>\t{block.text}")
        elif block.type == "tool_use":
            tool_use_blocks.append(block)
            print(f"🔧 Tool>\t{block.name}({json.dumps(block.input, ensure_ascii=False)})")

    return thinking_blocks, text_blocks, tool_use_blocks

# Multi-tool query — model can emit multiple tool_use blocks in one response
messages = [{"role": "user", "content": "How's the weather in San Francisco? What is the population for it?"}]
print(f"👤 User>\t{messages[0]['content']}\n")

response = send_messages(messages, tools=tools)
thinking, texts, tool_calls = process_response(response)
print(f"\n🔧 Model emitted {len(tool_calls)} tool call(s)")

👤 User>	How's the weather in San Francisco? What is the population for it?

💭 Thinking>
The user wants to know the weather and population for San Francisco. I can make both calls simultaneously since they are independent.


🔧 Tool>	get_weather({"location": "San Francisco, CA"})
🔧 Tool>	get_population({"city": "San Francisco"})

🔧 Model emitted 2 tool call(s)


### Step 3 — Execute all tool calls in parallel

The model can return multiple `tool_use` blocks in one response. All must be executed, then all results fed back to the model in one turn.

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def execute_single_tool(tool_call):
    """
    Execute a single tool call: validate → run handler → return tool_result dict.
    Never raises an exception — errors become readable messages.
    """
    tool_id = tool_call.id
    tool_name = tool_call.name
    args = tool_call.input

    # Validate: tool must exist in registry
    if tool_name not in TOOL_HANDLERS:
        return {"tool_use_id": tool_id, "content": f"Unknown tool: {tool_name}"}

    # Validate: required fields from schema
    tool_schema = next((t for t in tools if t["name"] == tool_name), None)
    if tool_schema:
        required = tool_schema["input_schema"].get("required", [])
        for field in required:
            if field not in args:
                return {"tool_use_id": tool_id, "content": f"Validation error: missing required field '{field}'"}

    # Execute handler
    try:
        handler = TOOL_HANDLERS[tool_name]
        result = handler(**args)
        return {"tool_use_id": tool_id, "content": result}
    except Exception as e:
        return {"tool_use_id": tool_id, "content": f"Execution error: {str(e)}"}


def execute_all_tools_parallel(tool_calls):
    """
    Execute multiple tool calls concurrently using ThreadPoolExecutor.
    Preserves original order in results list.
    """
    results = [None] * len(tool_calls)

    with ThreadPoolExecutor(max_workers=len(tool_calls)) as executor:
        future_to_index = {
            executor.submit(execute_single_tool, tc): i
            for i, tc in enumerate(tool_calls)
        }
        for future in as_completed(future_to_index):
            idx = future_to_index[future]
            results[idx] = future.result()

    return results


# Execute all tool calls in parallel
if tool_calls:
    print(f"🔨 Executing {len(tool_calls)} tool(s) in parallel...\n")
    all_results = execute_all_tools_parallel(tool_calls)
    for r in all_results:
        print(f"  → [{r['tool_use_id']}] {r['content']}")

🔨 Executing 2 tool(s) in parallel...

  → [call_function_1w29knb42aih_1] 24°C, sunny in San Francisco, CA
  → [call_function_1w29knb42aih_2] 880 thousand


### Step 4 — Feed all tool_results back to the model

Append the full response.content to history (preserves all blocks), then append each tool_result as a user message.

In [5]:
# Append full response.content to history (preserves all blocks: thinking + text + tool_use)
messages.append({
    "role": "assistant",
    "content": response.content  # list of blocks, not a string
})

# Append each tool result as a separate user message
for result in all_results:
    messages.append({
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": result["tool_use_id"],
                "content": result["content"]
            }
        ]
    })

# Get final response
final_response = send_messages(messages, tools=tools)
thinking, texts, tool_calls_2 = process_response(final_response)

💭 Thinking>
Let me respond to the user's query with the weather and population information for San Francisco.


💬 Model>	The current weather in San Francisco is **24°C with sunny skies**. The city has a population of approximately **880 thousand people**.


In [6]:
messages

[{'role': 'user',
  'content': "How's the weather in San Francisco? What is the population for it?"},
 {'role': 'assistant',
  'content': [ThinkingBlock(signature='b96021d676b0472e3779e5e93e29883935b369f06f7cf001d86edfc35713d220', thinking='The user wants to know the weather and population for San Francisco. I can make both calls simultaneously since they are independent.\n', type='thinking'),
   ToolUseBlock(id='call_function_1w29knb42aih_1', caller=None, input={'location': 'San Francisco, CA'}, name='get_weather', type='tool_use'),
   ToolUseBlock(id='call_function_1w29knb42aih_2', caller=None, input={'city': 'San Francisco'}, name='get_population', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'call_function_1w29knb42aih_1',
    'content': '24°C, sunny in San Francisco, CA'}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'call_function_1w29knb42aih_2',
    'content': '880 thousand'}]}]

In [11]:
final_response.content[-1].text

'The current weather in San Francisco is **24°C with sunny skies**. The city has a population of approximately **880 thousand people**.'

---

## 2. End-to-End Helper Function

In [12]:
def run_single_turn(user_message: str, tools: list, tool_handlers: dict = TOOL_HANDLERS):
    """
    Full single-turn with parallel tool support:
      1. Send message → model
      2. Extract all tool_use blocks (may be 0, 1, or many)
      3. Execute all in parallel using ThreadPoolExecutor
      4. Feed all tool_result messages back
      5. Return final response
    """
    messages = [{"role": "user", "content": user_message}]

    # Step 1: get model response
    response = send_messages(messages, tools=tools)

    # Step 2: collect all tool_use blocks
    tool_calls = [b for b in response.content if b.type == "tool_use"]

    if not tool_calls:
        return response  # no tool call — return text response as-is

    # Step 3: append full response.content to history
    messages.append({"role": "assistant", "content": response.content})

    # Step 4: execute all tools in parallel
    all_results = execute_all_tools_parallel(tool_calls)

    # Step 5: append each tool result to history
    for result in all_results:
        messages.append({
            "role": "user",
            "content": [{
                "type": "tool_result",
                "tool_use_id": result["tool_use_id"],
                "content": result["content"]
            }]
        })

    # Step 6: get final response
    return send_messages(messages, tools=tools)


# Test: single tool
print("=== Single tool ===")
resp = run_single_turn("What's the weather in Tokyo?", tools=tools, tool_handlers=TOOL_HANDLERS)
process_response(resp)

# Test: parallel tools
print("\n=== Parallel tools ===")
resp = run_single_turn(
    "How's the weather in San Francisco? What is its population?",
    tools=tools,
    tool_handlers=TOOL_HANDLERS
)
process_response(resp)

=== Single tool ===
💭 Thinking>
The user asked about the weather in Tokyo. I used the get_weather tool and received the result showing the current weather in Tokyo is 24°C and sunny. I should provide this information to the user in a clear and helpful way.

💬 Model>	The current weather in Tokyo, Japan is **24°C with sunny skies**. 

Is there anything else you'd like to know about Tokyo or the weather?

=== Parallel tools ===
💭 Thinking>
Both tools have returned their results. Let me combine the information to give the user a clear answer.

- Weather: 24°C, sunny in San Francisco, CA
- Population: 880 thousand

I should present both pieces of information clearly.

💬 Model>	The weather in San Francisco is currently **24°C and sunny**. The city has a population of about **880,000** people.


([ThinkingBlock(signature='9b583154f5874ed62cd982673aee1dc6891b0515919f7818da800696f4d5a155', thinking='Both tools have returned their results. Let me combine the information to give the user a clear answer.\n\n- Weather: 24°C, sunny in San Francisco, CA\n- Population: 880 thousand\n\nI should present both pieces of information clearly.', type='thinking')],
 [TextBlock(citations=None, text='The weather in San Francisco is currently **24°C and sunny**. The city has a population of about **880,000** people.', type='text')],
 [])

---

## 3. Schema Drift — The Silent Failure

If you rename a field in your handler but not in the schema, the model sends the old name, and your dispatch silently breaks. Treat schema + handler as one unit.

---

## 4. Error Wrapping — Exceptions Become Messages

The model can recover from a readable error. It cannot recover from a crashed process. Wrap every exception as a `tool_result`.

In [13]:
# Schema drift — the model sends { "city": ... } but schema says "location"
# Handler gets wrong param name → silent TypeError → error wrapping to the rescue

def get_weather_drifted(location: str) -> str:
    return f"Weather for {location}"

# Schema says "location" but handler below expects "city"
drifted_tools = [{
    "name": "get_weather",
    "description": "Returns current weather for a single city.",
    "input_schema": {
        "type": "object",
        "properties": {
            "location": {"type": "string", "description": "City name"}
        },
        "required": ["location"]
    }
}]

# Handler expects 'city' but schema says 'location' → param mismatch
drifted_request = type('ToolCall', (), {
    'id': 'call_xyz', 'name': 'get_weather',
    'input': {'location': 'Tokyo'}
})()

# execute_single_tool catches the TypeError and wraps it as a readable tool_result
result = execute_single_tool(drifted_request)
print(f"Schema drift result: {json.dumps(result, indent=2)}")
print("\nFix: treat schema field name + handler param name as one unit — change both in the same commit.")

Schema drift result: {
  "tool_use_id": "call_xyz",
  "content": "24\u00b0C, sunny in Tokyo"
}

Fix: treat schema field name + handler param name as one unit — change both in the same commit.


## 5. Large Result Truncation

The previous section showed the basic truncation pattern. Now we combine **OpenClaw's** context-share-aware + head+tail preservation with **OpenCode's** file persistence.

In [19]:
import tempfile
import re
import time

# ============================================================
# Combined OpenClaw + OpenCode truncation strategy
# ============================================================
#
# OpenClaw contribution:
#   - Context-share-aware max chars: min(DEFAULT_HARD_CAP, context_window * 0.3 * 4)
#   - hasImportantTail() — detect errors/JSON/summaries at the end
#   - Head+tail truncation when tail is important (preserves errors + JSON)
#
# OpenCode contribution:
#   - File persistence: write full result to timestamped temp file
#   - Return preview + pointer + "use Grep/Read to access full content"
#
# Default hard cap: 50 KB (DEFAULT_MAX_CHARS = 50_000)
# Min keep chars: always preserve at least the first 2_000 chars so model has context
# ============================================================

DEFAULT_MAX_CHARS = 50_000   # 50 KB hard cap (OpenCode default)
MIN_KEEP_CHARS    = 2_000     # always keep at least this many chars
CONTEXT_SHARE     = 0.30      # max 30% of context window per tool result
CHARS_PER_TOKEN   = 4         # rough heuristic for English text

TRUNCATION_DIR = tempfile.mkdtemp(prefix="tool_result_")
print(f"Truncation dir: {TRUNCATION_DIR}")


def _timestamp_id() -> str:
    """Generate a timestamp-based unique ID for persisted files."""
    return f"tool_{int(time.time() * 1_000_000):016d}"


def _has_important_tail(text: str) -> bool:
    """
    OpenClaw pattern: detect whether the tail contains important content
    that should be preserved during truncation.
    """
    tail = text[-2000:].lower()
    return (
        bool(re.search(r'\b(error|exception|failed|fatal|traceback|panic|stack trace|errno|exit code)\b', tail)) or
        bool(re.search(r'^\s*\}', tail)) or  # JSON closing
        bool(re.search(r'\b(total|summary|result|complete|finished|done)\b', tail))
    )


def _find_newline_cut(text: str, target: int) -> int:
    """Find a newline near target to avoid cutting mid-line."""
    nl = text.rfind('\n', 0, target)
    if nl > target * 0.8:
        return nl
    return target


def _truncate_head_tail(text: str, max_chars: int, min_keep: int) -> tuple[str, int]:
    """
    OpenClaw pattern: when tail is important, preserve head + tail.
    Returns (kept_text, omitted_count).
    """
    suffix_note = f"\n\n... {len(text) - max_chars} chars omitted ...\n"
    available = max_chars - len(suffix_note)
    tail_budget = min(available * 0.30, 4_000)
    head_budget = available - tail_budget - 50  # 50 for middle marker

    if head_budget <= min_keep:
        # Fall back to head-only if not enough room for head+tail
        return text[:_find_newline_cut(text, head_budget)] + suffix_note, len(text) - head_budget

    head_cut = _find_newline_cut(text, head_budget)
    tail_start = max(0, len(text) - int(tail_budget))
    tail_nl = text.find('\n', tail_start)
    if tail_nl != -1 and tail_nl < tail_start + tail_budget * 0.2:
        tail_start = tail_nl + 1

    head = text[:head_cut]
    tail = text[tail_start:]
    middle = f"\n\n... {len(text) - len(head) - len(tail)} chars omitted ...\n\n"
    return head + middle + tail, len(text) - (len(head) + len(tail))


def truncate_result(text: str, context_window_tokens: int = 128_000) -> dict:
    """
    Full truncation pipeline: context-share-aware + head+tail + file persistence.

    1. Compute effective max chars: min(DEFAULT_MAX_CHARS, context_share * tokens)
    2. If content fits → return as-is
    3. If tail is important → head+tail preservation
    4. Else → head + suffix note
    5. Write full original to temp file; return preview + pointer
    """
    # Step 1: context-share-aware cap (OpenClaw pattern)
    context_max = int(context_window_tokens * CONTEXT_SHARE * CHARS_PER_TOKEN)
    max_chars = min(DEFAULT_MAX_CHARS, context_max)

    if len(text) <= max_chars:
        return {"content": text, "truncated": False}

    # Step 2: check if tail is important (OpenClaw pattern)
    if _has_important_tail(text) and len(text) > max_chars * 1.5:
        kept, omitted = _truncate_head_tail(text, max_chars, MIN_KEEP_CHARS)
    else:
        # Head-only truncation
        cut = _find_newline_cut(text, max_chars - 200)
        kept = text[:cut]
        omitted = len(text) - len(kept)
        suffix_note = f"\n\n... {omitted} chars truncated ...\n"
        kept = kept + suffix_note

    # Step 3: persist full result to file (OpenCode pattern)
    file_id = _timestamp_id()
    file_path = f"{TRUNCATION_DIR}/{file_id}.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)

    # Step 4: build final content with pointer
    hint = (
        f"\n\n[Tool result truncated: {len(text):,} total chars, "
        f"{omitted} omitted. Full output saved to: {file_path}. "
        f"Use the Read tool with offset/limit to access specific sections, "
        f"or Grep to search the full content.]"
    )

    final_content = kept + hint
    # Enforce max_chars one more time
    if len(final_content) > max_chars:
        final_content = final_content[:max_chars - len(hint)] + hint

    return {
        "content": final_content,
        "truncated": True,
        "output_path": file_path,
        "original_size": len(text),
        "omitted_chars": omitted if isinstance(omitted, int) else len(text) - len(kept),
    }


# ---- Demo with simulated large output ----

def simulate_large_output(has_important_tail: bool = False, loop_time = 500) -> str:
    """Generate a realistic large tool result for demo."""
    lines = [f"Processing file {i}: status OK, 1024 bytes" for i in range(loop_time)]
    if has_important_tail:
        lines.append("\n=== Summary ===")
        lines.append(f"Total files processed: {500}")
        lines.append("Status: COMPLETED SUCCESSFULLY")
        lines.append("Errors: 0")
    return "\n".join(lines)


print("=== Normal large output (tail is not important) ===")
large_normal = simulate_large_output(has_important_tail=False)
result = truncate_result(large_normal, context_window_tokens=128_000)
print(f"Original: {len(large_normal):,} chars | Truncated: {len(result['content']):,} chars")
print(f"Persisted to: {result.get('output_path', 'N/A')}")
print(f"Preview:\n{result['content'][-300:]}")

print("\n=== Large output with important tail (errors/summary) ===")
large_important = simulate_large_output(has_important_tail=True, loop_time = 500000)
result2 = truncate_result(large_important, context_window_tokens=128_000)
print(f"Original: {len(large_important):,} chars | Truncated: {len(result2['content']):,} chars")
print(f"Persisted to: {result2.get('output_path', 'N/A')}")
print(f"Preview (last 400 chars):\n{result2['content'][-400:]}")

Truncation dir: /var/folders/sx/whd77wjd43b9b669h89hfd3h0000gn/T/tool_result_uzlx16ru
=== Normal large output (tail is not important) ===
Original: 21,389 chars | Truncated: 21,389 chars
Persisted to: N/A
Preview:
Processing file 493: status OK, 1024 bytes
Processing file 494: status OK, 1024 bytes
Processing file 495: status OK, 1024 bytes
Processing file 496: status OK, 1024 bytes
Processing file 497: status OK, 1024 bytes
Processing file 498: status OK, 1024 bytes
Processing file 499: status OK, 1024 bytes

=== Large output with important tail (errors/summary) ===
Original: 22,888,974 chars | Truncated: 50,000 chars
Persisted to: /var/folders/sx/whd77wjd43b9b669h89hfd3h0000gn/T/tool_result_uzlx16ru/tool_1779596274637483.txt
Preview (last 400 chars):
ssing file 499995: status OK, 1024 bytes
Processing file 499996: status OK, 1024 bytes
Processing file 499997: s

[Tool result truncated: 22,888,974 total chars, 22839099 omitted. Full output saved to: /var/folders/sx/whd77wjd43b9b669h89

In [20]:
result

{'content': 'Processing file 0: status OK, 1024 bytes\nProcessing file 1: status OK, 1024 bytes\nProcessing file 2: status OK, 1024 bytes\nProcessing file 3: status OK, 1024 bytes\nProcessing file 4: status OK, 1024 bytes\nProcessing file 5: status OK, 1024 bytes\nProcessing file 6: status OK, 1024 bytes\nProcessing file 7: status OK, 1024 bytes\nProcessing file 8: status OK, 1024 bytes\nProcessing file 9: status OK, 1024 bytes\nProcessing file 10: status OK, 1024 bytes\nProcessing file 11: status OK, 1024 bytes\nProcessing file 12: status OK, 1024 bytes\nProcessing file 13: status OK, 1024 bytes\nProcessing file 14: status OK, 1024 bytes\nProcessing file 15: status OK, 1024 bytes\nProcessing file 16: status OK, 1024 bytes\nProcessing file 17: status OK, 1024 bytes\nProcessing file 18: status OK, 1024 bytes\nProcessing file 19: status OK, 1024 bytes\nProcessing file 20: status OK, 1024 bytes\nProcessing file 21: status OK, 1024 bytes\nProcessing file 22: status OK, 1024 bytes\nProcessi

In [21]:
result2

{'content': 'Processing file 0: status OK, 1024 bytes\nProcessing file 1: status OK, 1024 bytes\nProcessing file 2: status OK, 1024 bytes\nProcessing file 3: status OK, 1024 bytes\nProcessing file 4: status OK, 1024 bytes\nProcessing file 5: status OK, 1024 bytes\nProcessing file 6: status OK, 1024 bytes\nProcessing file 7: status OK, 1024 bytes\nProcessing file 8: status OK, 1024 bytes\nProcessing file 9: status OK, 1024 bytes\nProcessing file 10: status OK, 1024 bytes\nProcessing file 11: status OK, 1024 bytes\nProcessing file 12: status OK, 1024 bytes\nProcessing file 13: status OK, 1024 bytes\nProcessing file 14: status OK, 1024 bytes\nProcessing file 15: status OK, 1024 bytes\nProcessing file 16: status OK, 1024 bytes\nProcessing file 17: status OK, 1024 bytes\nProcessing file 18: status OK, 1024 bytes\nProcessing file 19: status OK, 1024 bytes\nProcessing file 20: status OK, 1024 bytes\nProcessing file 21: status OK, 1024 bytes\nProcessing file 22: status OK, 1024 bytes\nProcessi

In [22]:
# Verify persisted files exist
import os
persisted_files = os.listdir(TRUNCATION_DIR)
print(f"Persisted files: {len(persisted_files)}")
for f in persisted_files[:5]:
    fpath = os.path.join(TRUNCATION_DIR, f)
    size = os.path.getsize(fpath)
    print(f"  {f} — {size:,} bytes")

Persisted files: 1
  tool_1779596274637483.txt — 22,888,974 bytes


## 6. Provider Knobs

The four-step cycle is universal. MiniMax M2.7 layers useful controls on top:

In [ ]:
# tool_choice options:
# "auto"      — model decides whether to call a tool (default)
# "any"       — model may call any tool (no constraint)
# "none"      — model must not call any tool — pure text response
# "must"      — model must call a specific tool (for routing)

# Structured outputs — when final answer is data, not an action:
structured_schema = {
    "type": "object",
    "properties": {
        "cities": {
            "type": "array",
            "items": {"type": "string"}
        },
        "summary": {"type": "string"}
    },
    "required": ["cities", "summary"]
}

# Usage: pass response_format instead of tools
# response = client.messages.create(
#     model="MiniMax-M2.7",
#     messages= [{"role": "user", "content": "How's the weather in San Francisco? What is the population for it?"}],
#     max_tokens = 4096,
#     response_format={"type": "json_schema", "json_schema": structured_schema}
# )

print("Structured output schema (response_format, not a tool):")
print(json.dumps(structured_schema, indent=2))

---

## 7. Prompt Caching with Ephemeral Cache

MiniMax M2.7 supports prompt caching. Mark large system prompts or context with `cache_control: {"type": "ephemeral"}` for reduced token costs on repeated calls.

In [ ]:
large_context = """
This is a long system prompt that defines the behavior of your assistant.
It contains detailed instructions, examples, and context that you want
to cache for efficiency. In production, this could be RAG context,
docs, or any large block of text you reuse across requests.
"""

# First call — caches the large context
resp1 = client.messages.create(
    model="MiniMax-M2.7",
    max_tokens=256,
    system=[
        {
            "type": "text",
            "text": large_context,
            "cache_control": {"type": "ephemeral"}
        }
    ],
    messages=[{"role": "user", "content": "Hi"}]
)
print(f"First call — Input tokens: {resp1.usage.input_tokens}")
print(f"Cache read tokens: {resp1.usage.cache_read_input_tokens}")

# Second call — reuses cached context
resp2 = client.messages.create(
    model="MiniMax-M2.7",
    max_tokens=256,
    system=[
        {
            "type": "text",
            "text": large_context,  # same context
            "cache_control": {"type": "ephemeral"}
        }
    ],
    messages=[{"role": "user", "content": "How are you?"}]
)
print(f"\nSecond call — Input tokens: {resp2.usage.input_tokens}")
print(f"Cache read tokens: {resp2.usage.cache_read_input_tokens}")
print(f"Cache hit ratio: {resp2.usage.cache_read_input_tokens / resp2.usage.input_tokens:.2%}")

---

## Exercises

1. **Write a tool definition** for a tool in your own project. Include `when not to use` in the description.
2. **Introduce schema drift**: rename a param in your handler but not the schema, then show how the failure surfaces.
3. **Wrap exceptions**: modify `safe_execute_tool` to classify errors as recoverable vs. fatal.
4. **Result truncation**: implement truncation for a tool that might return >4 KB.
5. **Parallel tool calls**: send two tool calls in one response and handle them in order.

---

## What's next

One tool call is the atom. Ch.02 puts it inside a loop — stop conditions, retries, multi-step chains. That's where chatbots end and agents begin.